In [3]:
from tensorflow.keras.models import load_model
from keras_self_attention import SeqSelfAttention
from tensorflow.keras.utils import custom_object_scope
import tensorflow.keras
import shap
import numpy as np
import os 
import glob
import pandas as pd
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import train_test_split
import tensorflow as tf

In [4]:
with custom_object_scope({'SeqSelfAttention': SeqSelfAttention}):
    load_model = tensorflow.keras.models.load_model('/Users/dimitrygrebenyuk/Yandex.Disk.localized/Working/PDF/Refinements/PDF-Cluster-Prediction/all_data/cifs/new_fold_3_model.hdf5')

In [5]:
os.chdir('/Users/dimitrygrebenyuk/Yandex.Disk.localized/Working/PDF/Refinements/PDF-Cluster-Prediction/all_data/cifs')
files = glob.glob('*.dat')


raw_data_points = []
with open('labels.txt', 'w') as labels:
    for f in files:
        # For files not containing 'processed' in their names (first type)
        df = pd.read_csv(f, usecols=[1], skiprows=201, header=None, delim_whitespace=True, skipfooter=800, engine='python')
        raw_data_points.append(df.values.ravel())
        if f[0] == 'p':
            labels.write('10')
            labels.write('\n')
        else:    
            labels.write(f[0])
            labels.write('\n')

raw_data_points = np.array(raw_data_points)

# Load the labels
labels = pd.read_csv("labels.txt", header=None)
labels = labels.values.ravel()  # convert the labels to a 1D array

normalize = Normalizer()
data_points = normalize.fit_transform(raw_data_points)

X_train, X_val, y_train, y_val = train_test_split(data_points, labels, test_size=0.2, random_state=42)

In [6]:
print("Train shape:", X_train[:50].shape)
print("Validation shape:", X_val[:10].shape)
print("Data type:", type(X_train))

Train shape: (50, 1000)
Validation shape: (10, 1000)
Data type: <class 'numpy.ndarray'>


In [7]:
import shap

try:
    explainer = shap.DeepExplainer(load_model, X_train[:50])
    shap_values = explainer.shap_values(X_val[:10])
    shap.summary_plot(shap_values, X_val[:10], plot_type="bar")
except Exception as e:
    print("An error occurred:", e) # Compute Shapley values for the first 10 test instances

shap.summary_plot(shap_values, X_val[:10], plot_type="bar")

keras is no longer supported, please use tf.keras instead.
Your TensorFlow version is newer than 2.4.0 and so graph support has been removed in eager mode and some static graphs may not be supported. See PR #1483 for discussion.
`tf.keras.backend.set_learning_phase` is deprecated and will be removed after 2020-10-11. To update it, simply pass a True/False value to the `training` argument of the `__call__` method of your layer or model.


Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089
An error occurred: in user code:

    File "/Users/dimitrygrebenyuk/opt/anaconda3/envs/shap/lib/python3.9/site-packages/shap/explainers/_deep/deep_tf.py", line 259, in grad_graph  *
        x_grad = tape.gradient(out, shap_rAnD)

    LookupError: gradient registry has no entry for: shap_BatchMatMulV2



NameError: name 'shap_values' is not defined